# AeroPulse Model 2 — Pollution Anomaly Detector

**Purpose:** detect pollution events by comparing current/fused PM2.5 with a historical cell baseline conditioned on time and meteorology, followed by an ML residual model.

LLD output: `anomaly_score`, `baseline_pm25`, `observed_pm25`, `residual`, `event_trigger`.

In [ ]:
!pip -q install pandas numpy scikit-learn lightgbm joblib requests pyarrow

In [ ]:
# Core
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

DATA_DIR = Path(os.getenv("AEROPULSE_DATA_DIR", "./data"))
ARTIFACT_DIR = Path(os.getenv("AEROPULSE_ARTIFACT_DIR", "./artifacts"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

REGION = {
    "min_lat": float(os.getenv("AEROPULSE_MIN_LAT", "28.0")),
    "max_lat": float(os.getenv("AEROPULSE_MAX_LAT", "32.8")),
    "min_lon": float(os.getenv("AEROPULSE_MIN_LON", "74.0")),
    "max_lon": float(os.getenv("AEROPULSE_MAX_LON", "78.5")),
}

OPENAQ_API_KEY = os.getenv("OPENAQ_API_KEY", "")
FIRMS_MAP_KEY = os.getenv("FIRMS_MAP_KEY", "")

def http_get_json(url, params=None, headers=None, timeout=60):
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def require_env(name):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value

print("AeroPulse ML environment initialized")
print("Region:", REGION)

In [ ]:
# OpenAQ v3 connector. OpenAQ exposes original measurements plus hourly aggregates.
# Use API key through OPENAQ_API_KEY.
OPENAQ_BASE = "https://api.openaq.org/v3"

def openaq_locations(bbox=REGION, limit=1000):
    params = {
        "bbox": f'{bbox["min_lon"]},{bbox["min_lat"]},{bbox["max_lon"]},{bbox["max_lat"]}',
        "limit": min(limit, 1000),
        "page": 1,
    }
    headers = {"X-API-Key": require_env("OPENAQ_API_KEY")}
    return http_get_json(f"{OPENAQ_BASE}/locations", params=params, headers=headers)

def openaq_location_hours(location_id, date_from, date_to, limit=1000):
    headers = {"X-API-Key": require_env("OPENAQ_API_KEY")}
    params = {
        "datetime_from": pd.Timestamp(date_from, tz="UTC").isoformat(),
        "datetime_to": pd.Timestamp(date_to, tz="UTC").isoformat(),
        "limit": limit,
        "page": 1,
    }
    return http_get_json(f"{OPENAQ_BASE}/locations/{int(location_id)}/hours",
                         params=params, headers=headers)

def discover_pm25_locations(max_locations=100):
    raw = openaq_locations(limit=max_locations)
    rows = raw.get("results", [])
    out = []
    for x in rows:
        params = x.get("parameters", [])
        names = {str(p.get("name", "")).lower() for p in params}
        if "pm25" in names or "pm2.5" in names:
            coords = x.get("coordinates") or {}
            if coords.get("latitude") is not None and coords.get("longitude") is not None:
                out.append({
                    "location_id": x["id"],
                    "name": x.get("name"),
                    "lat": coords["latitude"],
                    "lon": coords["longitude"],
                    "parameters": list(names),
                })
    return pd.DataFrame(out)

def normalize_openaq_hours(payload):
    rows = payload.get("results", [])
    out = []
    for r in rows:
        period = r.get("period", {})
        dt = (period.get("datetimeFrom") or {}).get("utc")
        param = r.get("parameter", {})
        out.append({
            "timestamp": pd.to_datetime(dt, utc=True, errors="coerce"),
            "parameter": str(param.get("name", "")).lower(),
            "value": safe_float(r.get("value")),
            "unit": r.get("unit"),
            "location_id": r.get("locationsId"),
        })
    return pd.DataFrame(out).dropna(subset=["timestamp"])

In [ ]:
# Open-Meteo is used here as a reproducible public weather/ERA5 access path.
# In production, replace this adapter with the IMD connector defined in the AeroPulse LLD.
OPEN_METEO_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
OPEN_METEO_FORECAST = "https://api.open-meteo.com/v1/forecast"

WEATHER_VARS = [
    "temperature_2m", "relative_humidity_2m", "surface_pressure",
    "precipitation", "wind_speed_10m", "wind_direction_10m",
    "boundary_layer_height"
]

def weather_archive(lat, lon, start_date, end_date):
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": pd.Timestamp(start_date).date().isoformat(),
        "end_date": pd.Timestamp(end_date).date().isoformat(),
        "hourly": ",".join(WEATHER_VARS),
        "timezone": "UTC",
    }
    return http_get_json(OPEN_METEO_ARCHIVE, params=params)

def weather_forecast(lat, lon, forecast_days=2):
    params = {
        "latitude": lat, "longitude": lon,
        "hourly": ",".join(WEATHER_VARS),
        "forecast_days": forecast_days,
        "timezone": "UTC",
    }
    return http_get_json(OPEN_METEO_FORECAST, params=params)

def normalize_weather(payload):
    h = payload.get("hourly", {})
    df = pd.DataFrame(h)
    if "time" in df:
        df["timestamp"] = pd.to_datetime(df.pop("time"), utc=True)
    return df

In [ ]:
def grid_id(lat, lon, resolution_km=1.0):
    # Approximate 1-km grid using latitude-dependent longitude scaling.
    lat_step = resolution_km / 111.32
    lon_step = resolution_km / (111.32 * np.cos(np.deg2rad(np.clip(lat, -89, 89))))
    ilat = np.floor(lat / lat_step).astype(int)
    ilon = np.floor(lon / lon_step).astype(int)
    return pd.Series(ilat).astype(str) + "_" + pd.Series(ilon).astype(str)

def add_grid(df, lat_col="lat", lon_col="lon"):
    out = df.copy()
    out["grid_id"] = grid_id(out[lat_col].astype(float), out[lon_col].astype(float))
    return out

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    dphi = np.deg2rad(lat2-lat1)
    dl = np.deg2rad(lon2-lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

## 1. Historical baseline data

The baseline must be learned without using the observation being scored. The notebook builds a robust median/quantile baseline by hour-of-week and then learns residual behavior from weather variables.

In [ ]:
locations = discover_pm25_locations(max_locations=100)
END = pd.Timestamp.utcnow().floor("h")
START = END - pd.Timedelta(days=int(os.getenv("TRAIN_DAYS", "60")))

records = []
for _, loc in locations.head(int(os.getenv("MAX_STATIONS", "30"))).iterrows():
    try:
        d = normalize_openaq_hours(
            openaq_location_hours(int(loc.location_id), START, END)
        )
        d = d[d.parameter.isin(["pm25","pm2.5"])].copy()
        if not d.empty:
            d["lat"], d["lon"], d["station_name"] = loc.lat, loc.lon, loc["name"]
            records.append(d)
    except Exception as e:
        print("failed", loc.location_id, e)

aq = pd.concat(records, ignore_index=True)
aq["pm25"] = pd.to_numeric(aq.value, errors="coerce")
aq = aq.dropna(subset=["pm25"])
aq["hour"] = aq.timestamp.dt.hour
aq["dow"] = aq.timestamp.dt.dayofweek
aq["hour_of_week"] = aq.dow * 24 + aq.hour
aq["grid_id"] = grid_id(aq.lat, aq.lon)
print(aq.shape)

In [ ]:
# Robust baseline. Production can store this as a versioned TimescaleDB feature table.
baseline = (
    aq.groupby(["location_id","hour_of_week"])["pm25"]
      .agg(
          baseline_pm25="median",
          q25=lambda s: s.quantile(0.25),
          q75=lambda s: s.quantile(0.75),
          mad=lambda s: np.median(np.abs(s - np.median(s)))
      )
      .reset_index()
)
aq = aq.merge(baseline, on=["location_id","hour_of_week"], how="left")
aq["residual"] = aq.pm25 - aq.baseline_pm25
aq["robust_z"] = aq.residual / (1.4826 * aq.mad.replace(0, np.nan))
aq["robust_z"] = aq.robust_z.replace([np.inf,-np.inf], np.nan).fillna(0)
display(aq[["timestamp","pm25","baseline_pm25","residual","robust_z"]].head())

## 2. ML residual model

The baseline explains normal seasonal/diurnal structure. The residual model explains how meteorology changes the expected deviation. The final anomaly score combines normalized residual magnitude with independent evidence rules.

This avoids making the ML model itself the sole event detector.

In [ ]:
weather_parts = []
for (lat, lon), _ in aq.groupby(["lat","lon"]):
    try:
        weather_parts.append(normalize_weather(weather_archive(lat, lon, START.date(), END.date())).assign(lat=lat, lon=lon))
    except Exception as e:
        print("weather", e)

weather = pd.concat(weather_parts, ignore_index=True)
aq2 = aq.merge(weather, on=["timestamp","lat","lon"], how="left")

features = [
    "temperature_2m","relative_humidity_2m","surface_pressure",
    "precipitation","wind_speed_10m","wind_direction_10m",
    "boundary_layer_height","hour","dow","lat","lon"
]
df = aq2.dropna(subset=["residual"]).copy()
cut = df.timestamp.quantile(0.80)
train, test = df[df.timestamp < cut], df[df.timestamp >= cut]

import joblib
from lightgbm import LGBMRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

imp = SimpleImputer(strategy="median")
Xtr = imp.fit_transform(train[features])
Xte = imp.transform(test[features])
res_model = LGBMRegressor(n_estimators=500, learning_rate=.03, num_leaves=31, random_state=SEED)
res_model.fit(Xtr, train.residual)
test["pred_residual"] = res_model.predict(Xte)
test["anomaly_residual"] = test.residual - test.pred_residual

print({
    "residual_MAE": mean_absolute_error(test.residual, test.pred_residual),
    "residual_RMSE": mean_squared_error(test.residual, test.pred_residual, squared=False),
    "residual_R2": r2_score(test.residual, test.pred_residual)
})

In [ ]:
# Calibrate an operational threshold on the validation period.
# Labels here are a proxy: large robust residuals. Replace with reviewed event labels in production.
test["proxy_event"] = (np.abs(test["robust_z"]) >= 3.0).astype(int)
test["anomaly_score"] = 1 - np.exp(-np.abs(test.anomaly_residual) / (test.pm25.std() + 1e-6))
test["pred_event"] = (test["anomaly_score"] >= 0.75).astype(int)

from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

validation = {
    "precision": precision_score(test.proxy_event, test.pred_event, zero_division=0),
    "recall": recall_score(test.proxy_event, test.pred_event, zero_division=0),
    "f1": f1_score(test.proxy_event, test.pred_event, zero_division=0),
    "confusion_matrix": confusion_matrix(test.proxy_event, test.pred_event).tolist()
}
validation

### Validation note

The LLD requires precision/recall/F1 and false-alert rate. The proxy label above is only for pipeline validation. A production anomaly detector must be evaluated against a reviewed event catalogue and independent evidence, not labels derived from the same residual calculation.

In [ ]:
bundle = {
    "model": res_model,
    "imputer": imp,
    "features": features,
    "baseline": baseline,
    "threshold": 0.75,
    "version": "1.0.0",
    "validation": validation
}
joblib.dump(bundle, ARTIFACT_DIR / "anomaly_detector.joblib")
baseline.to_parquet(ARTIFACT_DIR / "anomaly_baseline.parquet", index=False)
print("Saved anomaly model and baseline.")

## 3. Live anomaly scoring

The production path should load the latest historical baseline and score each new 1-km observation after evidence/quality checks.

In [ ]:
# Example live station scoring.
latest = []
for _, loc in locations.head(min(10,len(locations))).iterrows():
    try:
        p = http_get_json(
            f"{OPENAQ_BASE}/locations/{int(loc.location_id)}/latest",
            headers={"X-API-Key": require_env("OPENAQ_API_KEY")}
        )
        for r in p.get("results", []):
            if str((r.get("parameter") or {}).get("name","")).lower() in ("pm25","pm2.5"):
                dt = r.get("datetime") or r.get("period",{}).get("datetimeTo")
                dt = dt.get("utc") if isinstance(dt,dict) else dt
                latest.append({
                    "timestamp": pd.to_datetime(dt,utc=True,errors="coerce"),
                    "location_id": loc.location_id, "lat": loc.lat, "lon": loc.lon,
                    "observed_pm25": safe_float(r.get("value"))
                })
    except Exception as e:
        print(e)
live = pd.DataFrame(latest).dropna(subset=["timestamp","observed_pm25"])
display(live.head())

In [ ]:
if not live.empty:
    b = bundle["baseline"]
    live["hour_of_week"] = live.timestamp.dt.dayofweek*24 + live.timestamp.dt.hour
    live = live.merge(b[["location_id","hour_of_week","baseline_pm25","mad"]],
                      on=["location_id","hour_of_week"], how="left")
    live["residual"] = live.observed_pm25 - live.baseline_pm25
    live["robust_z"] = live.residual / (1.4826*live.mad.replace(0,np.nan))
    live["robust_z"] = live.robust_z.replace([np.inf,-np.inf],np.nan).fillna(0)

    wparts=[]
    for _, r in live.iterrows():
        try:
            w=normalize_weather(weather_forecast(r.lat,r.lon,forecast_days=1))
            w=w.iloc[(w.timestamp-r.timestamp).abs().argmin()].to_dict()
            wparts.append({**r.to_dict(), **w})
        except Exception:
            wparts.append(r.to_dict())
    scored=pd.DataFrame(wparts)

    X=scored[bundle["features"]]
    pred_res=bundle["model"].predict(bundle["imputer"].transform(X))
    scored["expected_residual"]=pred_res
    scored["anomaly_residual"]=scored["residual"]-pred_res
    scored["anomaly_score"]=1-np.exp(-np.abs(scored["anomaly_residual"])/(aq.pm25.std()+1e-6))
    scored["event_trigger"]=scored["anomaly_score"]>=bundle["threshold"]
    display(scored[["timestamp","location_id","observed_pm25","baseline_pm25","residual","anomaly_score","event_trigger"]])